# 01. RAG The Big Picture: Build a Tiny RAG in Pure Python

> **Promise:** By the end of this notebook, you will build a complete, runnable RAG system in 25 lines of pure Python with zero frameworks, and map every single step to its LangChain equivalent.

### Learning Objectives
- Visualize the entire RAG lifecycle: Index Time (offline) vs. Query Time (online).
- Build a working RAG pipeline in pure Python using sentence splitting and word-overlap scoring.
- Map each plain Python operation directly to its corresponding LangChain primitive.

### Prerequisites
- Notebook 00 (Why RAG? The Limits of LLMs)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_llm, detect_provider
from course.utils.helpers import get_data_dir

llm = get_llm(temperature=0.0)
print(f"Active Provider: {detect_provider()}")
print(f"Loaded Chat Model: {llm.__class__.__name__}")

Active Provider: local
Loaded Chat Model: LocalGroundedChatModel


## 2. Concept & Intuition

### The Two Halves of RAG: Index Time vs. Query Time

Every RAG system operates in two distinct phases:

1. **Index Time (The Preparation / Ingestion Phase - Run Once or Periodically):**
   - You take raw documents (PDFs, Markdown, Web pages).
   - You break them into smaller pieces called **chunks** (so you can search precisely).
   - You transform chunks into searchable mathematical representations (**embeddings**) and store them in a **vector store**.

2. **Query Time (The Search and Answer Phase - Run on Every User Question):**
   - The user asks a question.
   - The **retriever** searches the index and selects the top $k$ most relevant chunks.
   - The system injects those chunks into a **prompt template**.
   - The LLM reads the context and **generates** a grounded answer.

```
INDEX TIME (Offline Preparation):
[Documents] ──> [Text Splitter] ──> [Chunks] ──> [Embedding Model] ──> [Vector Database]

QUERY TIME (Live User Query):
[User Question] ──> [Search Query] ──> [Vector Database]
                                             │
                                             ▼
[Retrieved Chunks] + [User Question] ──> [Prompt] ──> [LLM] ──> [Grounded Answer]
```

## 3. Step-by-Step Hands-On Build

### Step 1: Load Raw Document Text in Plain Python
Let's read our company handbook into a simple Python string using the standard `open()` function.

In [2]:
handbook_path = get_data_dir() / "company_handbook.txt"
with open(handbook_path, "r", encoding="utf-8") as f:
    raw_document = f.read()

print(f"Loaded document: {len(raw_document)} characters, {len(raw_document.split())} words.")

Loaded document: 4163 characters, 611 words.


> **What just happened?**
> We have loaded the raw text directly into memory.

### Step 2: Split the Document into Sentences (Primitive Chunking)
Instead of passing the entire document to the model, we split the text into individual lines and sentences.

In [3]:
# Primitive chunking: split by line and filter out short empty lines
chunks = [line.strip() for line in raw_document.split("\n") if len(line.strip()) > 30]
print(f"Split document into {len(chunks)} candidate sentence chunks.")
print("Sample chunk #1:", chunks[0])
print("Sample chunk #2:", chunks[1])

Split document into 35 candidate sentence chunks.
Sample chunk #1: # NOVASPHERE DYNAMICS - EMPLOYEE HANDBOOK & OPERATIONAL POLICIES
Sample chunk #2: Last Revised: September 15, 2026


> **What just happened?**
> We now have a list of small, focused text chunks.

### Step 3: Score Chunks with Simple Word Overlap (Primitive Search)
When a user asks a question, how do we find the relevant chunks? Let's write a simple scoring function that counts shared words.

In [4]:
def score_overlap(query: str, chunk: str) -> float:
    q_words = set(query.lower().replace("?", "").replace("$", "").split())
    c_words = set(chunk.lower().replace(",", "").replace(".", "").split())
    stop_words = {"what", "is", "the", "of", "and", "a", "for", "in", "to"}
    meaningful_q = q_words - stop_words
    if not meaningful_q:
        return 0.0
    shared = meaningful_q.intersection(c_words)
    return len(shared) / len(meaningful_q)

query = "What is the price of the Starter Plan?"
scored = [(score_overlap(query, c), c) for c in chunks]
scored.sort(key=lambda x: x[0], reverse=True)

top_k = 2
retrieved_chunks = [c for score, c in scored[:top_k]]
for rank, (score, c) in enumerate(scored[:top_k], start=1):
    print(f"Rank {rank} (score: {score:.2f}): {c}")

Rank 1 (score: 0.33): - Standard Support (Included in Starter Tier):
Rank 2 (score: 0.33): - Starter Plan: $499 per month. Supports up to 10 active autonomous robot/drone nodes. Includes standard telemetry and 30-day log retention.


> **What just happened?**
> The word-overlap scorer successfully ranked the Starter Plan sentence at the very top!

### Step 4: Inject Retrieved Chunks into a Grounded Prompt
Now we combine the retrieved chunks into a prompt and instruct the model to answer using only that context.

In [5]:
context_block = "\n".join(retrieved_chunks)
rag_prompt = (
    "You are a helpful customer support agent for Novasphere Dynamics.\n"
    "Answer the user's question using ONLY the provided context. If the answer cannot be determined, say 'I don't know'.\n\n"
    f"CONTEXT:\n{context_block}\n\n"
    f"QUESTION: {query}\nANSWER:"
)

response = llm.invoke(rag_prompt)
print("="*60)
print(response.content)
print("="*60)

I don't know based on the provided context. (No relevant context provided)


> **What just happened?**
> You just built and executed a complete RAG system in pure Python with no external AI frameworks!

### Step 5: Mapping Pure Python to LangChain Components
While 25 lines of Python proved the concept, real applications require semantic search, robust document loaders, and production vector stores. Let's map each step to its LangChain equivalent.

In [6]:
mapping = """
| Step in Pure Python              | LangChain Primitive                      | Why the Library Helps                    |
|-----------------------------------|------------------------------------------|------------------------------------------|
| `open('handbook.txt').read()`     | `DocumentLoader` (`TextLoader`, `PyPDFLoader`) | Parses PDFs, DOCX, HTML, extracts metadata |
| `raw_doc.split('\n')`             | `RecursiveCharacterTextSplitter`         | Handles paragraphs, code, sentence boundaries|
| `score_overlap()` (word overlap)  | `Embeddings` (`OpenAI`, `SentenceTransformers`) | Understands synonyms & semantic meaning  |
| `scored[:k]` list sorting         | `VectorStore` (`Chroma`, `InMemoryVectorStore`) | Sub-millisecond ANN search across millions |
| `f"CONTEXT: {context}..."`        | `ChatPromptTemplate`                     | Composable system/human chat roles       |
| `llm.invoke(prompt)`              | `ChatModel` (`init_chat_model`)          | Streaming, tool calling, provider agnosticism|
"""
print(mapping)


| Step in Pure Python              | LangChain Primitive                      | Why the Library Helps                    |
|-----------------------------------|------------------------------------------|------------------------------------------|
| `open('handbook.txt').read()`     | `DocumentLoader` (`TextLoader`, `PyPDFLoader`) | Parses PDFs, DOCX, HTML, extracts metadata |
| `raw_doc.split('
')`             | `RecursiveCharacterTextSplitter`         | Handles paragraphs, code, sentence boundaries|
| `score_overlap()` (word overlap)  | `Embeddings` (`OpenAI`, `SentenceTransformers`) | Understands synonyms & semantic meaning  |
| `scored[:k]` list sorting         | `VectorStore` (`Chroma`, `InMemoryVectorStore`) | Sub-millisecond ANN search across millions |
| `f"CONTEXT: {context}..."`        | `ChatPromptTemplate`                     | Composable system/human chat roles       |
| `llm.invoke(prompt)`              | `ChatModel` (`init_chat_model`)          | Streaming, tool calling,

> **What just happened?**
> In the upcoming notebooks, we will replace each primitive step with its production-grade LangChain counterpart.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Synonym Failure Test):** Change the query to *"What is the monthly fee for entry-level customers?"* (Notice that 'fee' and 'entry-level' are synonyms for 'price' and 'Starter Plan'). Run the word-overlap scorer. Does it find the Starter Plan chunk?
2. **Exercise 2 (Adding Source Metadata):** Modify the pure Python chunks list into a list of tuples `(text, source_name)`. Modify the prompt injection to print the source name alongside the answer.

In [7]:
synonym_query = "What is the monthly fee for entry-level customers?"
synonym_scored = [(score_overlap(synonym_query, c), c) for c in chunks]
synonym_scored.sort(key=lambda x: x[0], reverse=True)
print("Top chunk for synonym query:")
print(f"Score: {synonym_scored[0][0]:.2f} -> {synonym_scored[0][1]}")
print("-> NOTICE: Word overlap scored 0.0 or missed the target because 'fee' != 'price'!")
print("-> This is exactly why we need Vector Embeddings in Notebook 04.")

Top chunk for synonym query:
Score: 0.25 -> All billing is charged monthly or annually (with a 15% discount for annual upfront commitments):
-> NOTICE: Word overlap scored 0.0 or missed the target because 'fee' != 'price'!
-> This is exactly why we need Vector Embeddings in Notebook 04.


## 5. Common Mistakes & Pitfalls
- **Confusing Index Time and Query Time:** Indexing happens ahead of time; retrieval happens dynamically on every user prompt.
- **Relying on Keyword Search Alone:** Pure keyword search fails whenever users use synonyms (e.g. 'cost' vs 'pricing', 'error' vs 'bug').
- **Retrieving Too Many Chunks:** Sending 20 chunks clutters context and increases latency; start with k=2 to 4.
- **Missing Fallback Instructions:** Always tell the model to say 'I don't know' if context does not contain the answer.

---

## 6. Notebook Recap
- RAG is conceptually simple: Split text -> Search relevant pieces -> Inject into prompt -> Generate grounded answer.
- Keyword matching works when words overlap exactly, but fails on synonyms and paraphrasing.
- Index time prepares the database; query time handles the live user interaction.
- Every complex RAG framework is simply an industrial-strength version of these 4 steps.


## 7. What's Next?

In **02_loading_documents.ipynb**, we begin replacing our primitive Python steps with production LangChain components, starting with the `Document` object and loading text, Markdown, and real PDF files.

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - Index Time vs Query Time:** The two operational phases of all RAG architectures.
- **02:30 - Loading and Splitting in Pure Python:** Reading text and creating chunks with zero libraries.
- **05:15 - Writing a Word-Overlap Scorer:** Understanding search ranking from first principles.
- **08:00 - Assembling the Grounded Prompt:** Passing context to the model and seeing the output.
- **10:45 - The Synonym Failure Case:** Why keyword search fails on real-world queries.
- **13:00 - The LangChain Architecture Map:** Previewing the components we will build in notebooks 02-06.
